# A script for combining responses into a single table with file name validation

Algorithm:
   * Copy the first CSV file to the resulting file, including metadata.
   * Rename column `predicted_text` to the CSV filename.
   * From the second CSV file to the end of the list, merge source and destination files.

In [1]:
import os
import shutil
import pandas as pd

from itertools import islice


csv_list = [
    "test_base_qwen.csv",
]

vit_gpt2_versions = [1, 2, 3, 4, 5, 6, 7]
for v in vit_gpt2_versions:
    csv_list.append(f"test_vit-gpt2_transformer_v0{v}.csv")

csv_result = "test_all.csv"  # resulting single table


def merge_csv(source, destination):
    """ Merge source and destination CSV files by the column name. """
    src = pd.read_csv(source)       # take data from here
    dst = pd.read_csv(destination)  # add data here

    # Check number of rows
    assert len(src) == len(dst), f"Error: Different number of rows! {len(src)} vs {len(dst)}"

    # Check filename matches line by line
    is_equal = (src["filename"] == dst["filename"]).all()
    
    if is_equal:
        print(f"✅ '{source}' merged with '{destination}'")
        dst[source] = src["predicted_text"]
        dst.to_csv(destination, index=False)
    else:
        mismatch_idx = (src["filename"] != dst["filename"]).idxmax()
        raise ValueError(
            f"🛑 Error! Mismatch in string number {mismatch_idx}\n"
            f"\tin the file '{source}' : '{src.loc[mismatch_idx, 'filename']}'\n"
            f"\tin the file '{destination}' : '{dst.loc[mismatch_idx, 'filename']}'"
        )


def show(csv_file):
    """ Show first rows of CSV file """
    print(f"\nFile: {csv_file}")
    df = pd.read_csv(csv_file)
    display(df.head(3))  # show first rows


for csv_file in csv_list:
    show(csv_file)


File: test_base_qwen.csv


,filename,actual_text,predicted_text
0,14_4_154037.dcm_norm.png,Pulmonary emphysema and pneumosclerosis. Aorti...,Frontal chest radiograph demonstrates clear lu...
1,39_2_59975.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,Frontal chest radiograph demonstrates clear lu...
2,68_5_106546.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,Normal chest radiograph demonstrating clear lu...



File: test_vit-gpt2_transformer_v01.csv


,filename,actual_text,predicted_text
0,14_4_154037.dcm_norm.png,Pulmonary emphysema and pneumosclerosis. Aorti...,aortic arch is sclerotic. Heart is enlarged to...
1,39_2_59975.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,-related changes. Scoliosis.
2,68_5_106546.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,aortic arch is sclerotic. Heart is enlarged to...



File: test_vit-gpt2_transformer_v02.csv


...

,filename,actual_text,predicted_text
0,14_4_154037.dcm_norm.png,Pulmonary emphysema and pneumosclerosis. Aorti...,"clinical findings are present, pneumonia in th..."
1,39_2_59975.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,-up imaging after treatment. Right-sided pneum...
2,68_5_106546.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,"clinical findings are present, pneumonia in th..."



File: test_vit-gpt2_transformer_v06.csv


,filename,actual_text,predicted_text
0,14_4_154037.dcm_norm.png,Pulmonary emphysema and pneumosclerosis. Aorti...,-up imaging after treatment. Increased pulmona...
1,39_2_59975.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,-defined opacity in the right upper lobe. Scol...
2,68_5_106546.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,-shaped scoliosis. Lungs are clear without vis...



File: test_vit-gpt2_transformer_v07.csv


,filename,actual_text,predicted_text
0,14_4_154037.dcm_norm.png,Pulmonary emphysema and pneumosclerosis. Aorti...,-shaped scoliosis. Pulmonary hila are well-def...
1,39_2_59975.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,post-tuberculous changes. Pulmonary emphysema....
2,68_5_106546.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,-shaped scoliosis. Lungs are clear without vis...


In [2]:
shutil.copy2(csv_list[0], csv_result)  # copy 1st file, including metadata

# Rename column "predicted_text" to CSV filename
df = pd.read_csv(csv_result)
df.rename(columns={"predicted_text": csv_list[0]}, inplace=True)
df.to_csv(csv_result, index=False)
print(f"✅ '{csv_list[0]}' copied to '{csv_result}'")

for csv_file in csv_list[1:]:
    merge_csv(csv_file, csv_result)

show(csv_result)  # show resulting file

✅ 'test_base_qwen.csv' copied to 'test_all.csv'
✅ 'test_vit-gpt2_transformer_v01.csv' merged with 'test_all.csv'
✅ 'test_vit-gpt2_transformer_v02.csv' merged with 'test_all.csv'
✅ 'test_vit-gpt2_transformer_v03.csv' merged with 'test_all.csv'
✅ 'test_vit-gpt2_transformer_v04.csv' merged with 'test_all.csv'
✅ 'test_vit-gpt2_transformer_v05.csv' merged with 'test_all.csv'
✅ 'test_vit-gpt2_transformer_v06.csv' merged with 'test_all.csv'
✅ 'test_vit-gpt2_transformer_v07.csv' merged with 'test_all.csv'

File: test_all.csv


,filename,actual_text,test_base_qwen.csv,test_vit-gpt2_transformer_v01.csv,test_vit-gpt2_transformer_v02.csv,test_vit-gpt2_transformer_v03.csv,test_vit-gpt2_transformer_v04.csv,test_vit-gpt2_transformer_v05.csv,test_vit-gpt2_transformer_v06.csv,test_vit-gpt2_transformer_v07.csv
0,14_4_154037.dcm_norm.png,Pulmonary emphysema and pneumosclerosis. Aorti...,Frontal chest radiograph demonstrates clear lu...,aortic arch is sclerotic. Heart is enlarged to...,aortic arch sclerosis.,internal medicine consultation recommended. Pu...,s are clear without focal or infiltrative opac...,"clinical findings are present, pneumonia in th...",-up imaging after treatment. Increased pulmona...,-shaped scoliosis. Pulmonary hila are well-def...
1,39_2_59975.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,Frontal chest radiograph demonstrates clear lu...,-related changes. Scoliosis.,aortic arch sclerosis. Scoliosis.,internal medicine consultation recommended. Pu...,posteroanterior chest X-ray. Pulmonary emphyse...,-up imaging after treatment. Right-sided pneum...,-defined opacity in the right upper lobe. Scol...,post-tuberculous changes. Pulmonary emphysema....
2,68_5_106546.dcm_norm.png,Pulmonary emphysema. Pneumosclerosis. Aortic s...,Normal chest radiograph demonstrating clear lu...,aortic arch is sclerotic. Heart is enlarged to...,aortic arch sclerosis. Scoliosis.,-shaped opacity in the lower lobe on the left....,posteroanterior chest X-ray. Pulmonary emphyse...,"clinical findings are present, pneumonia in th...",-shaped scoliosis. Lungs are clear without vis...,-shaped scoliosis. Lungs are clear without vis...
